# 怎样预测未来一周用电量并滚动评价？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'forecast_windows': 8, 'forest_mae': 6.138293515018776, 'seasonal_mae': 6.8722398243423095, 'empirical_coverage': 0.9642857142857143, 'interval_width': 30.10406630928395, 'nominal_target': 0.9, 'iid_guarantee_claimed': False}


## 1. 补全：分预测距离 MAE
计算各 horizon 的 MAE，并核对 7 个距离都有 8 个预测。

In [2]:
by_horizon=result.groupby('horizon')['absolute_error'].agg(['mean','count'])
assert len(by_horizon)==7 and by_horizon['count'].eq(8).all()
print(by_horizon)

             mean  count
horizon                 
1        6.793379      8
2        7.052337      8
3        7.490665      8
4        6.608478      8
5        6.719422      8
6        3.513985      8
7        4.789789      8


## 2. 修改：只改校准目标
用原校准误差的 95% 分位数构造区间，核对宽度不缩小；这是预设目标对照，不按测试选宽度。

In [3]:
wide=float(np.quantile(calibration_errors,0.95,method='higher'))
assert wide>=width
new_coverage=np.mean(abs(result['actual']-result['forest'])<=wide)
print({'区间宽度':2*wide,'测试经验覆盖':float(new_coverage)})

{'区间宽度': 31.477873107871176, '测试经验覆盖': 0.9642857142857143}


## 3. 迁移：第 731 天的新预测起点
在 origin=730 生成未来 7 天，核对模型输出和基线形状，并计算这段已知真值的误差。

In [4]:
pred,base,actual=forecast_at(730)
assert pred.shape==base.shape==actual.shape==(7,)
print({'模型MAE':float(np.mean(abs(pred-actual))),'基线MAE':float(np.mean(abs(base-actual)))})

{'模型MAE': 8.569449873332568, '基线MAE': 6.568973545629502}
